In [1]:
%pip install soundfile
%pip install huggingface_hub
%pip install datasets
%pip install librosa 
%pip install torch
%pip install torchcodec

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [75]:
# Imports
import soundfile as sf
import IPython
import numpy as np
import matplotlib.pyplot as plt
from glob import glob
import pickle
import sklearn
import torch
import torchcodec
from datasets import load_dataset
from datasets import Audio


In [76]:
# Dataset was already split into train, test, validation. Split = "all" combines the dataset
dataset = load_dataset("Julian-br/GuitarJam", split = "all")


Resolving data files:   0%|          | 0/467 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/59 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/54 [00:00<?, ?it/s]

In [79]:
dataset = dataset.cast_column("audio", Audio(decode=False))
# Store the duration of each audio clip
durations = []

for sample in dataset:
    audio_info = sample["audio"]
    info = sf.info(audio_info["path"])
    durations.append(info.duration)

print(f"Minimum duration: {np.min(durations):.2f} seconds")
print(f"Maximum duration: {np.max(durations):.2f} seconds")
print(f"Mean duration: {np.mean(durations):.2f} seconds")

# split long clips to get a similar length in each

def audio_splitter(audio_array, num_clips) :
    samples_per_clip = len(audio_array) // num_clips
    split_audio = []
        
    for i in range(num_clips):

        # Calculate the starting sample
        start = i * samples_per_clip

        # Calculate the ending sample
        if i == num_clips - 1:
            end = len(audio_array)
        else:
            end = (i + 1) * samples_per_clip

        # Extract the audio section
        clip = audio_array[start:end]
        split_audio.append(clip)
    return split_audio


new_dataset = []

for sample in dataset:
    
    # Load audio
    audio_path = sample["audio"]["path"]
    audio_array, sr = sf.read(audio_path)

    # Calculate duration
    duration = len(audio_array) / sr
    # Split into 2 or 3 new clips depending on size
    if (duration > 20) and (duration < 30) :
        clips = audio_splitter(audio_array, 2)

        # Append both clips separately
        for clip in clips:
            new_dataset.append(clip)
    elif duration > 30 :
        clips = audio_splitter(audio_array, 3)
        for clip in clips:
            new_dataset.append(clip)
    else:
        # Keep original recording
        new_dataset.append(audio_array)

Minimum duration: 2.10 seconds
Maximum duration: 42.38 seconds
Mean duration: 15.55 seconds


In [84]:
x = new_dataset[620]
IPython.display.Audio(data = x, rate=sr)